# Notebook 8.1  Dialect identification, read honestly

*A classifier on self-supervised embeddings, its confusion matrix, and a small spoken-document search.*

---

*Companion notebook to* **Introduction to Arabic Speech Technology** *by Hend S. Al-Khalifa.*

**How to run.** Open this notebook in Google Colab or run it locally with the
pinned environment in `requirements.txt`. Every notebook in this series runs end
to end with **no downloads and no accounts**: where a real corpus or a
pretrained model is unavailable, a clearly marked fallback stands in for it, and
the notebook says which path it took. Cells that need a download are marked
`OPTIONAL` and are safe to skip.

**On data.** Where you substitute a real corpus, record its release version and
its licence in the provenance cell at the end. A result without them is not
reproducible, which is the habit this book asks for in every chapter.

<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/notebooks-archive/ch08_dialect_id.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## What this notebook does

A dialect identifier, read honestly, and a spoken-document search demo.

1. Represent each clip as an utterance embedding from a self-supervised encoder, and train a light classifier on top. This is the standard recipe and it is two lines once the embeddings exist.
2. Report accuracy **and** macro-averaged F1, because a corpus with an MSA majority makes accuracy look good for the wrong reason.
3. Read the confusion matrix. The chapter's argument is that neighbouring varieties on the continuum are the hard cases, and the matrix is where that becomes visible rather than assertable.
4. Index a small set of transcribed clips and rank them against a keyword, with a morphological expansion, which is the smallest honest version of spoken document retrieval.

A synthetic fallback generates embeddings with the geometry the chapter
describes, so everything runs with no downloads. Where to plug in real
embeddings is marked.

In [ ]:
NOTEBOOK = 'ch08_dialect_id.ipynb'

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)

rng = np.random.default_rng(7)
DIALECTS = ['MSA', 'Gulf', 'Levantine', 'Egyptian', 'Maghrebi']
print('ready')

## 1. Embeddings

Real embeddings come from a pretrained encoder: mean-pool its hidden states
over time and you have one vector per utterance. The OPTIONAL cell shows the
call.

The fallback below places each dialect at a point in a 64-dimensional space,
and the geography is the lesson: the varieties are laid out along a line, in
the order they sit on the continuum, so neighbours are genuinely close and the
classifier's mistakes will be between neighbours rather than uniform. That is
what the chapter means by a continuum rather than a set of categories.

In [ ]:
def synthetic_embeddings(per_class=120, dim=64, spread=1.0):
    # the dialects are placed along a line: neighbours are near, ends are far
    centres = np.zeros((len(DIALECTS), dim))
    for i in range(len(DIALECTS)):
        centres[i, :8] = rng.normal(size=8) * 0.2 + i * 0.9
    X, y = [], []
    for i in range(len(DIALECTS)):
        X.append(centres[i] + rng.normal(scale=spread, size=(per_class, dim)))
        y += [i] * per_class
    return np.vstack(X), np.array(y)


X, y = synthetic_embeddings()
n_train = int(0.7 * len(y))
order = rng.permutation(len(y))
train, test = order[:n_train], order[n_train:]
print(f'{X.shape[0]} utterances, {X.shape[1]} dimensions, '
      f'{len(DIALECTS)} dialects')
print(f'{len(train)} train, {len(test)} test')

In [ ]:
# OPTIONAL: real embeddings from a self-supervised encoder.
REAL_AUDIO = []       # [(path, dialect), ...]

if REAL_AUDIO:
    import torch
    import soundfile as sf
    from transformers import AutoModel, AutoFeatureExtractor

    name = 'facebook/wav2vec2-xls-r-300m'
    fx = AutoFeatureExtractor.from_pretrained(name)
    enc = AutoModel.from_pretrained(name).eval()
    vectors, labels = [], []
    for path, dialect in REAL_AUDIO:
        wav, sr = sf.read(path)
        inputs = fx(wav, sampling_rate=sr, return_tensors='pt')
        with torch.no_grad():
            hidden = enc(**inputs).last_hidden_state
        vectors.append(hidden.mean(dim=1).squeeze().numpy())
        labels.append(DIALECTS.index(dialect))
    X, y = np.array(vectors), np.array(labels)
    print(f'{len(X)} real embeddings')
else:
    print('skipped: using the synthetic embeddings above')

## 2. The classifier, and two numbers instead of one

Accuracy is the share of clips labelled correctly. On a corpus where most of
the audio is Modern Standard Arabic, a classifier that says MSA to everything
scores well on it.

Macro-averaged F1 gives every dialect the same weight regardless of how much
audio it has, so a variety that fails completely drags it down. The gap between
the two numbers is a measure of how unbalanced the test set is, and reporting
only the first is the pitfall the chapter names.

In [ ]:
clf = LogisticRegression(max_iter=2000)
clf.fit(X[train], y[train])
pred = clf.predict(X[test])

print(f'accuracy          : {accuracy_score(y[test], pred):.3f}')
print(f'macro-averaged F1 : {f1_score(y[test], pred, average="macro"):.3f}\n')
print(classification_report(y[test], pred, target_names=DIALECTS, digits=3))

# the majority-class baseline every claim has to beat
majority = np.bincount(y[train]).argmax()
base = accuracy_score(y[test], np.full_like(y[test], majority))
print(f'a classifier that always says {DIALECTS[majority]}: '
      f'accuracy {base:.3f}, macro F1 '
      f'{f1_score(y[test], np.full_like(y[test], majority), average="macro"):.3f}')

## 3. The confusion matrix

Read along the rows. A dialect confused with its neighbour on the continuum is
the expected failure and tells you the model has learned something real. A
dialect confused with a distant one usually means a data problem: a mislabelled
speaker, a channel the model is keying on, or a class with too few speakers to
generalise from.

In [ ]:
cm = confusion_matrix(y[test], pred, normalize='true')
fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(DIALECTS)), DIALECTS, rotation=30, ha='right')
ax.set_yticks(range(len(DIALECTS)), DIALECTS)
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('confusion, normalized by row')
for i in range(len(DIALECTS)):
    for j in range(len(DIALECTS)):
        if cm[i, j] > 0.005:
            ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center',
                    color='white' if cm[i, j] > 0.5 else 'black', fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

off = [(DIALECTS[i], DIALECTS[j], cm[i, j])
       for i in range(len(DIALECTS)) for j in range(len(DIALECTS)) if i != j]
off.sort(key=lambda x: -x[2])
print('the three largest confusions:')
for a, b, v in off[:3]:
    neighbour = abs(DIALECTS.index(a) - DIALECTS.index(b)) == 1
    print(f'  {a} taken for {b}: {v:.1%}'
          f'{"   (neighbours on the continuum)" if neighbour else ""}')

## 4. Spoken document search

Once clips are transcribed, finding one is a text problem with an Arabic
complication: the word in the query is rarely written the way it appears in the
transcript. It carries a prefix, or a pronoun is attached, or the dialect
spells it differently.

The expansion below is crude on purpose, so you can see exactly what it does.
The point is the difference between the two result lists.

In [ ]:
CLIPS = [
    ('c1', 'وين اقرب محطة بنزين', 'Gulf'),
    ('c2', 'المحطة قريبة من المكتبة', 'MSA'),
    ('c3', 'ذهبت الى محطة القطار صباحا', 'MSA'),
    ('c4', 'محطتنا بعيدة شوي', 'Gulf'),
    ('c5', 'الطقس حار اليوم', 'MSA'),
    ('c6', 'وصلنا للمحطة متأخرين', 'Levantine'),
]
PREFIXES = ['و', 'ف', 'ب', 'ك', 'ل', 'ال', 'وال', 'بال', 'لل', 'فال']
SUFFIXES = ['ها', 'هم', 'نا', 'كم', 'ه', 'ك', 'ي', 'ات', 'ة', 'تنا']


def variants(word):
    out = {word}
    for p in PREFIXES:
        out.add(p + word)
        for s in SUFFIXES:
            out.add(p + word + s)
    for s in SUFFIXES:
        out.add(word + s)
    return out


def search(query, expand=False):
    keys = variants(query) if expand else {query}
    hits = []
    for cid, text, dialect in CLIPS:
        score = sum(1 for w in text.split() if w in keys)
        if score:
            hits.append((score, cid, dialect, text))
    return sorted(hits, reverse=True)


QUERY = 'محطة'
for expand in (False, True):
    print(f'query {QUERY!r}, expansion {"on" if expand else "off"}:')
    hits = search(QUERY, expand)
    for score, cid, dialect, text in hits:
        print(f'   {cid}  [{dialect:<9}] {text}')
    if not hits:
        print('   nothing found')
    print()
print('The expansion reaches the clip written with the definite article and '
      'the one with a preposition fused to it. It still misses محطتنا, where '
      'attaching the possessive turned the ta marbuta into an ordinary ta: '
      'concatenating affixes is not morphology, and this is where a real '
      'system needs an analyser rather than a list. Expansion also lets in '
      'false matches, and both have to be measured.')

## 5. What to report

Not the accuracy alone. The chapter's Reproducibility Note asks for the dialect
taxonomy and the per-class counts, because two papers that both say Gulf may
not mean the same thing, and a class with two speakers in it is not a class.

In [ ]:
counts = np.bincount(y[test], minlength=len(DIALECTS))
print(f'{"dialect":<12} {"test clips":>11} {"speakers":>10}  taxonomy note')
for i, d in enumerate(DIALECTS):
    print(f'{d:<12} {counts[i]:>11} {"fill in":>10}  '
          f'which regions this label covers')
print('\nAlso report: the encoder and its revision, the layer the embeddings '
      'were taken from, whether the split is speaker-disjoint, and the audio '
      'condition, because a classifier can learn the channel instead of the '
      'dialect and score well doing it.')

## Provenance

Fill this in before you quote any number from this notebook. It is the same
information the chapter's Reproducibility Note asks for, and it is the
difference between a result and a screenshot.

In [ ]:
PROVENANCE = {
    'notebook': NOTEBOOK,
    'ran_on': 'fill in the date you ran it',
    'data': 'corpus name and release version, or "synthetic fallback"',
    'licence': 'the licence of the data you used',
    'model': 'model name and revision, or "none"',
    'normalization': 'the normalization applied before scoring',
    'hardware': 'CPU or the GPU model',
}
for k, v in PROVENANCE.items():
    print(f'{k:>15}: {v}')